# Previous Config

## Packages and folders

In [ ]:
!pip install -U ranx sentence-transformers datasets accelerate #xformers
!pip uninstall -y wandb

import argparse
import itertools
import json
import os
import re
import time
import tempfile
import unicodedata
import shutil
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ranx import Qrels, Run, evaluate
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, InputExample, losses, util, evaluation
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset
from transformers import EarlyStoppingCallback
from huggingface_hub import list_models, model_info, snapshot_download, login, whoami
from google.colab import userdata

import psutil
import multiprocessing as mp
import traceback
import gc
from multiprocessing import Process, Queue
import signal
import sys

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 19.9 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=642cdae2ffc34b6ed863

In [ ]:
token = userdata.get('HF_TOKEN')
login(token=token)
print(whoami())

{'type': 'user', 'id': '663b5ba4347ccbda6657011c', 'name': 'davidpedidos', 'fullname': 'Garcia', 'isPro': False, 'avatarUrl': '/avatars/c7ec7e4f116a9a483fdbdb95dc009fc2.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab_exe', 'role': 'fineGrained', 'createdAt': '2025-12-03T15:52:39.743Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '663b5ba4347ccbda6657011c', 'type': 'user', 'name': 'davidpedidos'}, 'permissions': []}]}}}}


In [ ]:
# Configuración básica
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
today = time.strftime("%Y-%m-%d")

In [ ]:
def set_directories():
  project_dir = Path(__name__).resolve().parents[1]
  project_dir = project_dir / 'content' / 'TalentCLEF-TaskA'
  date_dir = project_dir.parent / 'output' / today
  date_dir.mkdir(parents=True, exist_ok=True)

  # Crear subdirectorio incremental por ejecución (001, 002, ...)
  exec_dirs = sorted([d for d in date_dir.iterdir() if d.is_dir() and d.name.isdigit() and len(d.name) == 3])
  if exec_dirs and not any(exec_dirs[-1].iterdir()):
      output_dir = exec_dirs[-1]
  else:
      next_id = int(exec_dirs[-1].name) + 1 if exec_dirs else 1
      output_dir = date_dir / f"{next_id:03d}"
      output_dir.mkdir(exist_ok=True)

  return project_dir, output_dir

project_dir, output_dir = set_directories()

In [ ]:
!git clone https://github.com/davidgc14/TalentCLEF-TaskA.git
!cp ./TalentCLEF-TaskA/src/output/ranking_spanish_validation.csv {output_dir.parent.parent}
!rm -r sample_data/

Cloning into 'TalentCLEF-TaskA'...
remote: Enumerating objects: 870, done.
remote: Counting objects: 100% (656/656), done.
remote: Compressing objects: 100% (333/333), done.
remote: Total 870 (delta 381), reused 577 (delta 309), pack-reused 214 (from 1)
Receiving objects: 100% (870/870), 58.18 MiB | 5.05 MiB/s, done.
Resolving deltas: 100% (448/448), done.


## Evaluation functions

In [ ]:
def load_qrels(qrels_path):
    """
    Loads the qrels file (TREC format: q_id, iter, doc_id, rel)
    and converts it to a Qrels object.
    """
    qrels_df = pd.read_csv(qrels_path, sep="\t", header=None,
                           names=["q_id", "iter", "doc_id", "rel"],
                           dtype={"q_id": str, "doc_id": str, "rel":int})

    return Qrels.from_df(qrels_df, q_id_col="q_id", doc_id_col="doc_id", score_col="rel")

def load_run(run_path):
    """
    Loads the run file (TREC format: q_id, Q0, doc_id, rank, score, [tag])
    and converts it to a Run object.
    """
    run_df = pd.read_csv(run_path, sep=r"\s+", header=None)

    # Assign column names based on the number of columns
    if run_df.shape[1] == 5:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score"]
    elif run_df.shape[1] >= 6:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score", "tag"]
    else:
        raise ValueError("The run file does not have the expected format.")

    run_df["q_id"] = run_df.q_id.astype(str)
    run_df["doc_id"] = run_df.doc_id.astype(str)
    return Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

def evaluate_run(qrels_path, run_path):
    """Evalúa un run y devuelve los resultados como dict."""
    qrels = load_qrels(qrels_path)
    run = load_run(run_path)
    metrics = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]
    return evaluate(qrels, run, metrics)

In [ ]:

# ========================
# DATA LOADING AND ENCODING
# ========================

def load_spanish_data(data_dir):
    """Load queries and corpus elements from Spanish data directory."""
    queries_path = data_dir / "queries"
    corpus_elements_path = data_dir / "corpus_elements"

    queries = pd.read_csv(queries_path, sep="\t")
    corpus_elements = pd.read_csv(corpus_elements_path, sep="\t")

    return (
        queries.q_id.to_list(),
        queries.jobtitle.to_list(),
        corpus_elements.c_id.to_list(),
        corpus_elements.jobtitle.to_list(),
    )


def encode_data(model, queries_texts, corpus_texts, model_name, device):
    """Encode queries and corpus using the model."""
    print('Encoding data:', model_name, 'on device:', device)
    query_embeddings = model.encode(queries_texts, convert_to_tensor=True, show_progress_bar=False, device=device)
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=False, device=device)
    # Convert to float32 to ensure compatibility with cosine similarity if fp16 is enabled
    return query_embeddings.to(torch.float32), corpus_embeddings.to(torch.float32)


def calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name):
    """Calculate cosine similarities and format results in TREC format."""
    print('Calculating similarities and preparing results...')
    similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

    results = []
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities[q_idx])  # Orden descendente
        for rank, c_idx in enumerate(sorted_indices):
            doc_id = corpus_ids[c_idx]
            score = similarities[q_idx, c_idx]
            results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} {model_name}")
    return results


def run_evaluation_temp(qrels_path, results, model_name):
    """Run evaluation using temporary file without saving."""
    print('Evaluating Spanish monolingual performance...')

    # Crear archivo temporal
    with tempfile.NamedTemporaryFile(mode='w', suffix='.trec', delete=False, encoding='utf-8') as tmp_file:
        tmp_file.write("\n".join(results))
        tmp_path = tmp_file.name

    try:
        evaluation_results = evaluate_run(qrels_path, tmp_path)
    finally:
        # Eliminar archivo temporal
        os.unlink(tmp_path)

    return evaluation_results


def get_model_name(model):
    """Extract model name from the model object."""
    model_name = model[0].auto_model.config._name_or_path
    return model_name.split("/")[-1]


# ========================
# EVALUATION FUNCTION
# ========================

def spanish_monolingual_evaluation(model, device, source):
    """Evaluate model performance on Spanish monolingual data."""
    data_dir = project_dir / 'data' / source / 'spanish'
    qrels_path = data_dir / "qrels.tsv"

    print('Loading Spanish data...')
    queries_ids, queries_texts, corpus_ids, corpus_texts = load_spanish_data(data_dir)
    model_name = get_model_name(model)

    query_embeddings, corpus_embeddings = encode_data(model, queries_texts, corpus_texts, model_name, device)
    results = calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name)

    evaluation_results = run_evaluation_temp(qrels_path, results, model_name)
    print('Spanish evaluation completed')
    return evaluation_results


# ========================
# SAVING RESULTS
# ========================

def save_spanish_results(evaluation_results, model_name, nickname, source):
    """Save Spanish monolingual evaluation results to JSON file."""
    print("Saving Spanish evaluation results...")

    json_path = output_dir / "results_spanish_monolingual.json"

    results_data = {
        "metadata": {
            "type": "spanish_monolingual",
            "model_name": model_name,
            "nickname": nickname,
            "source": source,
            "timestamp": today
        },
        "results": evaluation_results
    }

    with open(json_path, "w", encoding="utf-8") as jf:
        json.dump(results_data, jf, indent=2, ensure_ascii=False)

    print(f"Saved Spanish results to {json_path}")


# ========================
# RANKING
# ========================

def update_spanish_ranking(map_score, model_name, nickname, source):
    """Update Spanish-specific ranking CSV file with current execution results."""
    print("Updating Spanish ranking file...")

    ranking_file = output_dir.parent.parent / f"ranking_spanish_{source}.csv"
    execution_id = output_dir.name

    new_record = {
        'timestamp': today,
        'execution_id': execution_id,
        'model_name': model_name,
        'model_alias': nickname,
        'map_es_es': map_score
    }

    if ranking_file.exists():
        df = pd.read_csv(ranking_file)
    else:
        df = pd.DataFrame(columns=['timestamp', 'execution_id', 'model_name', 'model_alias', 'map_es_es'])

    # Double check for existing identical record
    comparison_cols = ['model_name', 'model_alias', 'map_es_es']

    if not df.empty:
        new_record_comparison = {k: new_record.get(k, np.nan) for k in comparison_cols}
        existing_records = df[comparison_cols].to_dict('records')

        for existing in existing_records:
            if all(abs(existing.get(k, np.nan) - new_record_comparison.get(k, np.nan)) < 1e-4
                   if isinstance(new_record_comparison.get(k), (float, int)) and not np.isnan(new_record_comparison.get(k, np.nan))
                   else existing.get(k) == new_record_comparison.get(k)
                   for k in comparison_cols):
                print("Ya existe un registro idéntico en el ranking español. No se agregará el nuevo registro.")
                return

    # Evitar warning de pandas con DataFrame vacío
    if df.empty:
        df = pd.DataFrame([new_record])
    else:
        df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)

    df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
    df.to_csv(ranking_file, index=False)

    print(f"Ranking español actualizado en {ranking_file}")


# ========================
# MAIN PIPELINE
# ========================

def run_spanish_evaluation(model_name, nickname, device, source):
    """Run complete Spanish monolingual evaluation pipeline."""
    model = SentenceTransformer(model_name, device=device)

    evaluation_results = spanish_monolingual_evaluation(model, device, source)
    save_spanish_results(evaluation_results, model_name, nickname, source)

    map_score = evaluation_results.get('map', np.nan)
    update_spanish_ranking(map_score, model_name, nickname, source)

    print(f"\n{'='*50}")
    print(f"Evaluación completada para {model_name}")
    print(f"MAP español-español: {map_score:.4f}")
    print(f"{'='*50}\n\n")

# Training path

## Config

In [ ]:
MODEL_NAME = 'Linq-AI-Research/Linq-Embed-Mistral'

In [ ]:
BATCH_SIZE = 16
EPOCHS = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Rutas de salida para el entrenamiento Hugging Face
model_output_path = output_dir / 'finetuned_models' / MODEL_NAME
best_model_path = model_output_path / 'best_model'
logs_path = model_output_path / 'logs'

print(f"Directorio de proyecto: {project_dir}")
print(f"Guardando outputs en: {model_output_path}")

Directorio de proyecto: /content/TalentCLEF-TaskA
Guardando outputs en: /content/output/2025-12-23/001/finetuned_models/Linq-AI-Research/Linq-Embed-Mistral


## Data preparation

### Training data

In [ ]:
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFC', text)
    # Reemplazar puntuación por espacio para no pegar palabras
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
training_file = project_dir / 'data' / 'training' / 'spanish' / 'taskA_training_es.tsv'

train_df = pd.read_csv(training_file, sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

In [ ]:
expanded = []
for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']

    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]

    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                't1': a,
                't2': b
            })

df_expanded = pd.DataFrame(expanded)

In [ ]:
# Normalización
df_expanded['t1'] = df_expanded['t1'].apply(normalize_text)
df_expanded['t2'] = df_expanded['t2'].apply(normalize_text)

# Eliminar duplicados (considerando pares desordenados)
# Creamos una columna temporal 'key' para filtrar
df_expanded['key'] = df_expanded.apply(lambda x: frozenset([x['t1'], x['t2']]), axis=1)
df_unique = df_expanded.drop_duplicates(subset='key').drop(columns=['key'])

# Filtrar vacíos
df_unique = df_unique[(df_unique['t1'] != "") & (df_unique['t2'] != "")]

# Convertir las columnas a string ANTES de crear el Dataset
df_unique['t1'] = df_unique['t1'].astype(str)
df_unique['t2'] = df_unique['t2'].astype(str)

# Resetear índice y crear el Dataset sin preservar el índice de pandas
df_unique = df_unique.reset_index(drop=True)
train_df_for_ds = df_unique[['t1', 't2']].rename(columns={'t1': 'sentence_1', 't2': 'sentence_2'})

In [ ]:
train_dataset = Dataset.from_pandas(train_df_for_ds, preserve_index=False)
print(f"Total de parejas en entrenamiento: {len(train_dataset)}")

Total de parejas en entrenamiento: 17855


### Validation data

In [ ]:
validation_dir = project_dir / 'data' / 'validation' / 'spanish'

corpus_val = pd.read_csv(validation_dir / 'corpus_elements', sep='\t')
queries_val = pd.read_csv(validation_dir / 'queries', sep='\t')
qrels_val = pd.read_csv(validation_dir / 'qrels.tsv', sep='\t', header=None)
qrels_val.columns = ['q_id', 'iter', 'doc_id', 'rel']

In [ ]:
# Diccionarios normalizados
corpus_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in corpus_val.iterrows()}
queries_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in queries_val.iterrows()}

# Construir dataset de pares positivos para calcular la Loss de validación
val_pairs = []
for _, row in qrels_val.iterrows():
    q_id = str(row.iloc[0])
    c_id = str(row.iloc[2])
    score = int(row.iloc[3])

    if score > 0 and q_id in queries_dict and c_id in corpus_dict:
        val_pairs.append({
            'sentence_1': queries_dict[q_id],
            'sentence_2': corpus_dict[c_id]
        })

In [ ]:
val_dataset = Dataset.from_list(val_pairs)
print(f"Pares de validación para Early Stopping: {len(val_dataset)}")

Pares de validación para Early Stopping: 7579


## Model preparation

In [ ]:
# Cargar modelo
model = SentenceTransformer(MODEL_NAME, device=DEVICE, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Definir función de pérdida
# MultipleNegativesRankingLoss es estándar para pares (Anchor, Positive)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Definir argumentos de entrenamiento (usando la clase nativa de SentenceTransformers/HF)
args = SentenceTransformerTrainingArguments(
    output_dir=str(model_output_path),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    fp16=(DEVICE == 'cuda'),  # Usar precisión mixta si hay GPU
    eval_strategy="epoch",    # Evaluar al final de cada época
    save_strategy="epoch",    # Guardar checkpoint al final de cada época
    load_best_model_at_end=True, # Cargar el mejor modelo al terminar (CRUCIAL para Early Stopping)
    metric_for_best_model="eval_loss",
    save_total_limit=2,       # No llenar el disco con checkpoints
    logging_steps=50,
    batch_sampler=BatchSamplers.NO_DUPLICATES # Evitar duplicados en batch para MNRLoss
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=0,  # Número de épocas sin mejora antes de detener
    early_stopping_threshold=0 # Umbral mínimo de mejora
)

# Inicializar Trainer con EarlyStoppingCallback
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    callbacks=[early_stopping_callback]
)

RuntimeError: Bad StatusOr access: RESOURCE_EXHAUSTED: Error allocating device buffer: Attempting to allocate 224.00M. That was not possible. There are 41.41M free.; (0x0x0_HBM0)

In [ ]:
print('Resultados antes del entrenamiento')
spanish_monolingual_evaluation(model, DEVICE, 'validation')

## Training model

In [ ]:
print("\n--- Comenzando entrenamiento... ---")
trainer.train()
print("\n--- Entrenamiento finalizado ---")

In [ ]:
# final_model = SentenceTransformer(str('/content/output/2025-12-17/002/finetuned_models/Lajavaness/bilingual-embedding-large/checkpoint-1116'), device=DEVICE, trust_remote_code=True)
# spanish_monolingual_evaluation(final_model, DEVICE, 'validation')

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
logs = trainer.state.log_history
df = pd.DataFrame(logs)

# Loss de entrenamiento
train_loss_df = df[df["loss"].notna()][["epoch", "loss"]]

# Loss de validación
eval_loss_df = df[df["eval_loss"].notna()][["epoch", "eval_loss"]]

plt.figure()
plt.plot(train_loss_df["epoch"], train_loss_df["loss"], label="Train Loss")
plt.plot(eval_loss_df["epoch"], eval_loss_df["eval_loss"], label="Validation Loss")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss Function in Fine-Tunning")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(f"Guardando mejor modelo en: {best_model_path}")
model.save(str(best_model_path))

shutil.make_archive(best_model_path, 'zip', best_model_path)

final_model = SentenceTransformer(str(best_model_path), device=DEVICE, trust_remote_code=True)

# Evaluation and saving

## Final evaluation

In [ ]:
spanish_monolingual_evaluation(final_model, DEVICE, 'validation')